# Project B — Telco Customer Churn
EDA, preprocessing, training and evaluation of five scikit-learn classifiers (Logistic Regression, Decision Tree, KNN, Gaussian NB, SVM) for binary churn prediction, followed by a final validation of the saved files.

- Dataset: https://www.kaggle.com/datasets/blastchar/telco-customer-churn (not included in this repository)
- Run in the BITS Pilani Virtual Lab: Python 3.14, scikit-learn 1.8.0
- Outputs saved to `models/`: five `.joblib` pipelines, `metadata.json`, `test_set.csv`, `metrics_project_b.csv`


In [3]:
# =========================================================
# PROJECT B — Telco Customer Churn: EDA + TRAINING (single cell)
# Same structure as Project A. Outputs are kept in their own
# folder (models_project_b) so nothing from Project A is overwritten.
# =========================================================

# ---------------------------------------------------------
# SETUP & IMPORTS
# ---------------------------------------------------------
import pandas as pd
from pathlib import Path
from IPython.display import display

# Show all columns when printing dataframes
pd.set_option("display.max_columns", None)

# Set the path to the folder containing the dataset
data_dir = Path("/drive/notebooks")
csv_path = data_dir / "Telco-Customer-Churn.csv"

# Quick check of what files are in the directory
print("Files in directory:")
for f in sorted(data_dir.iterdir()):
    print(f" - {f.name} ({f.stat().st_size / 1024:.1f} KB)")


# ---------------------------------------------------------
# LOAD DATA
# ---------------------------------------------------------
df = pd.read_csv(csv_path)
print(f"\nDataset shape: {df.shape[0]} rows, {df.shape[1]} columns")

target = "Churn"
id_col = "customerID"
features = [col for col in df.columns if col not in (target, id_col)]
print(f"Predictive features (customerID and Churn excluded): {len(features)}")

display(df.head())


# ---------------------------------------------------------
# DATA TYPES, MISSING VALUES & BLANKS
# ---------------------------------------------------------
df.info()

print("\n--- Missing Values (true NaN) ---")
missing = df.isna().sum()
if missing.any():
    print(missing[missing > 0])
else:
    print("No NaN values found.")

# A cell holding only a space is NOT NaN to pandas, so check text cells separately
print("\n--- Blank / whitespace-only cells ---")
blank_counts = df.apply(lambda col: col.astype(str).str.strip().eq("")).sum()
print(blank_counts[blank_counts > 0] if blank_counts.any() else "None")


# ---------------------------------------------------------
# DUPLICATES
# ---------------------------------------------------------
print("\n--- Duplicates ---")
print(f"Duplicate customerIDs: {df[id_col].duplicated().sum()}")
print(f"Exact duplicate rows: {df.duplicated().sum()}")

# Rows identical on all 19 features (different customers with the same plan)
print(f"Extra rows identical on all features (ignoring ID): {df.duplicated(subset=features).sum()}")
conflicts = df[df.duplicated(subset=features, keep=False)].groupby(features)[target].nunique()
print(f"Groups of identical features with different labels: {conflicts[conflicts > 1].count()}")


# ---------------------------------------------------------
# TOTALCHARGES: INVESTIGATE BLANKS, THEN CONVERT TO NUMERIC
# ---------------------------------------------------------
print("\n--- TotalCharges investigation ---")
blank_rows = df[df["TotalCharges"].astype(str).str.strip() == ""]
print(f"Blank TotalCharges rows: {len(blank_rows)}")
display(blank_rows[[id_col, "tenure", "MonthlyCharges", "Contract", target]])

# errors="coerce" turns anything non-numeric (the blanks) into NaN instead of crashing
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print(f"NaN after conversion: {df['TotalCharges'].isna().sum()}")
print(f"Rows with tenure == 0: {(df['tenure'] == 0).sum()}")


# ---------------------------------------------------------
# TARGET DISTRIBUTION
# ---------------------------------------------------------
print("\n--- Target Variable Breakdown ---")
class_counts = df[target].value_counts()
class_pct = (df[target].value_counts(normalize=True) * 100).round(2)

dist_df = pd.DataFrame({"Count": class_counts, "Percentage (%)": class_pct})
print(dist_df)

imbalance = class_counts.max() / class_counts.min()
print(f"\nImbalance ratio (majority to minority): {imbalance:.2f}x")


# ---------------------------------------------------------
# CATEGORICAL & NUMERICAL FEATURES
# ---------------------------------------------------------
# Decided by meaning, not storage type: SeniorCitizen is stored as 0/1 but is a yes/no category
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
cat_cols = [col for col in features if col not in num_cols]

print(f"\nNumerical columns ({len(num_cols)}): {num_cols}")
print(f"Categorical columns ({len(cat_cols)}): {cat_cols}")

print("\nCategory levels per column:")
for col in cat_cols:
    print(f"  {col}: {df[col].value_counts().to_dict()}")

stats = df[num_cols].describe().T
stats["skewness"] = df[num_cols].skew().round(2)
display(stats.round(2))


# ---------------------------------------------------------
# DATA LEAKAGE & STRONG SIGNALS
# ---------------------------------------------------------
print("\n--- Leakage Check ---")
suspicious_keywords = {"id", "customerid", "churn_flag", "churned"}
suspicious_cols = [c for c in features if c.lower() in suspicious_keywords]
print(f"Suspicious columns among features: {suspicious_cols if suspicious_cols else 'None'}")

# Is TotalCharges just tenure x MonthlyCharges? (redundancy, not leakage)
ratio = df["TotalCharges"] / (df["tenure"] * df["MonthlyCharges"])
print(f"\nTotalCharges / (tenure x MonthlyCharges): median {ratio.median():.3f}, "
      f"min {ratio.min():.3f}, max {ratio.max():.3f}")

# Spearman = Pearson correlation on ranks (no SciPy needed)
churn_flag = (df[target] == "Yes").astype(int)
correlations = df[num_cols].rank().corrwith(churn_flag.rank()).round(3)
print("\nSpearman correlation of numeric features with churn:")
print(correlations)

print("\nChurn rate by key categories:")
for col in ["Contract", "InternetService", "PaymentMethod"]:
    rates = df.groupby(col)[target].apply(lambda s: round((s == "Yes").mean(), 3))
    print(f"  {col}: {rates.to_dict()}")


print("\n" + "=" * 70 + "\nSTAGE 3 — PREPROCESSING, TRAINING, EVALUATION\n" + "=" * 70)


# ---------------------------------------------------------
# P0 — IMPORTS, VERSIONS, PATHS
# ---------------------------------------------------------
print('\n--- P0 ---')
import sys, json, time
import numpy as np, sklearn, joblib

print("Python", sys.version.split()[0], "| pandas", pd.__version__,
      "| scikit-learn", sklearn.__version__, "| joblib", joblib.__version__)
RANDOM_STATE = 42
DATA_PATH = csv_path                                  # reuses the path from the EDA part
MODEL_DIR = Path("models_project_b"); MODEL_DIR.mkdir(exist_ok=True)   # separate from Project A


# ---------------------------------------------------------
# P1 — CLEAN: drop customerID, fix TotalCharges, encode target
# ---------------------------------------------------------
print('\n--- P1 ---')
df = pd.read_csv(DATA_PATH)
TARGET = "Churn"
CLASS_NAMES = {0: "No churn", 1: "Churn"}
print("Raw shape:", df.shape)

df = df.drop(columns=["customerID"])                               # an identifier, not a predictor
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)   # 11 blanks = tenure 0, never billed
df[TARGET] = (df[TARGET] == "Yes").astype(int)                     # Yes -> 1 (positive class), No -> 0
print("Clean shape:", df.shape, "| TotalCharges NaN left:", int(df["TotalCharges"].isna().sum()))
print(df[TARGET].value_counts().sort_index().rename(index=CLASS_NAMES))


# ---------------------------------------------------------
# P2 — STRATIFIED 80/20 TRAIN/TEST SPLIT
# ---------------------------------------------------------
print('\n--- P2 ---')
from sklearn.model_selection import train_test_split

X = df.drop(columns=[TARGET]); y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print("Train:", X_train.shape, "| Test:", X_test.shape)
split = pd.DataFrame({"train": y_train.value_counts().sort_index(),
                      "test": y_test.value_counts().sort_index()})
split["train %"] = (split["train"] / split["train"].sum() * 100).round(2)
split["test %"] = (split["test"] / split["test"].sum() * 100).round(2)
display(split.rename(index=CLASS_NAMES))


# ---------------------------------------------------------
# P3 — DEFINE THE FIVE PIPELINES
# ---------------------------------------------------------
print('\n--- P3 ---')
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, PowerTransformer, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV

NUM_COLS = ["tenure", "MonthlyCharges", "TotalCharges"]
CAT_COLS = [c for c in X.columns if c not in NUM_COLS]

def make_preprocessor(numeric_step):
    """One-hot encode the 16 categorical columns; treat the 3 numeric ones with numeric_step."""
    onehot = OneHotEncoder(handle_unknown="ignore", drop="if_binary", sparse_output=False)
    return ColumnTransformer([("cat", onehot, CAT_COLS), ("num", numeric_step, NUM_COLS)])

models = {
    "Logistic Regression": Pipeline([("prep", make_preprocessor(StandardScaler())),
        ("clf", LogisticRegression(max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE))]),
    "Decision Tree": Pipeline([("prep", make_preprocessor("passthrough")),
        ("clf", DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE))]),
    "KNN": Pipeline([("prep", make_preprocessor(StandardScaler())),
        ("clf", KNeighborsClassifier(n_neighbors=5))]),
    "Gaussian NB": Pipeline([("prep", make_preprocessor(PowerTransformer(method="yeo-johnson", standardize=True))),
        ("clf", GaussianNB())]),
    "SVM": Pipeline([("prep", make_preprocessor(StandardScaler())),
        ("clf", CalibratedClassifierCV(SVC(kernel="rbf", class_weight="balanced", random_state=RANDOM_STATE),
                                       method="sigmoid", cv=5, ensemble=False))]),
}
for name, pipe in models.items():
    print(f"{name:20s} ->", " -> ".join(pipe.named_steps))


# ---------------------------------------------------------
# P4 — TRAIN ON TRAINING SET ONLY
# ---------------------------------------------------------
print('\n--- P4 ---')
for name, pipe in models.items():
    t0 = time.perf_counter()
    pipe.fit(X_train, y_train)
    n_cols = pipe.named_steps["prep"].transform(X_train.head(1)).shape[1]
    print(f"{name:20s} trained in {time.perf_counter() - t0:.2f} s | "
          f"classes: {[int(c) for c in pipe.classes_]} | encoded columns: {n_cols}")


# ---------------------------------------------------------
# P5 — EVALUATE ON THE SAME TEST SET (binary, positive class = Churn)
# ---------------------------------------------------------
print('\n--- P5 ---')
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef)

def evaluate(pipe, X_te, y_te):
    y_pred = pipe.predict(X_te)
    churn_proba = pipe.predict_proba(X_te)[:, 1]      # column 1 = probability of class 1 (Churn)
    return {"Accuracy": accuracy_score(y_te, y_pred),
            "AUC": roc_auc_score(y_te, churn_proba),
            "Precision": precision_score(y_te, y_pred, pos_label=1, zero_division=0),
            "Recall": recall_score(y_te, y_pred, pos_label=1, zero_division=0),
            "F1": f1_score(y_te, y_pred, pos_label=1, zero_division=0),
            "MCC": matthews_corrcoef(y_te, y_pred)}

results = pd.DataFrame({name: evaluate(pipe, X_test, y_test) for name, pipe in models.items()}).T
display(results.round(4))


# ---------------------------------------------------------
# P6 — SAVE MODELS, METADATA, TEST SET, METRICS
# ---------------------------------------------------------
print('\n--- P6 ---')
for name, pipe in models.items():
    fname = MODEL_DIR / (name.lower().replace(" ", "_") + ".joblib")
    joblib.dump(pipe, fname)
    print(f"Saved {fname} ({fname.stat().st_size / 1024:.1f} KB)")

meta = {"target": TARGET, "class_names": CLASS_NAMES, "positive_class": 1,
        "feature_columns": list(X.columns), "numeric_columns": NUM_COLS, "categorical_columns": CAT_COLS,
        "cleaning": "drop customerID; TotalCharges to numeric with blanks filled as 0; Churn Yes=1 No=0",
        "random_state": RANDOM_STATE, "test_size": 0.2, "sklearn_version": sklearn.__version__,
        "models": {n: n.lower().replace(" ", "_") + ".joblib" for n in models}}
(MODEL_DIR / "metadata.json").write_text(json.dumps(meta, indent=2))
X_test.assign(**{TARGET: y_test}).to_csv(MODEL_DIR / "test_set.csv", index=False)
results.to_csv(MODEL_DIR / "metrics_project_b.csv")
print("Saved metadata.json, test_set.csv, metrics_project_b.csv")


# ---------------------------------------------------------
# P7 — RELOAD SAVED MODELS AND VERIFY
# ---------------------------------------------------------
print('\n--- P7 ---')
for name, pipe in models.items():
    loaded = joblib.load(MODEL_DIR / (name.lower().replace(" ", "_") + ".joblib"))
    same = (loaded.predict(X_test) == pipe.predict(X_test)).all()
    print(f"{name:20s} reload OK, identical predictions: {same}")


Files in directory:
 - Intro.ipynb (42.0 KB)
 - Lorenz.ipynb (6.4 KB)
 - Mandelbrot.ipynb (7.1 KB)
 - ProjectB.ipynb (0.1 KB)
 - Telco-Customer-Churn.csv (954.6 KB)
 - Untitled Folder (0.0 KB)
 - cpp-third-party-libs.ipynb (6.8 KB)
 - cpp-tiny-ray-tracer.ipynb (13.3 KB)
 - cpp.ipynb (15.2 KB)
 - fetal_health.csv (223.4 KB)
 - models (0.0 KB)
 - models_project_b (0.0 KB)
 - r.ipynb (8.6 KB)
 - sqlite.ipynb (6.9 KB)

Dataset shape: 7043 rows, 21 columns
Predictive features (customerID and Churn excluded): 19


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

,customerID,tenure,MonthlyCharges,Contract,Churn
488,4472-LVYGI,0,52.55,Two year,No
753,3115-CZMZD,0,20.25,Two year,No
936,5709-LVOEQ,0,80.85,Two year,No
1082,4367-NUYAO,0,25.75,Two year,No
1340,1371-DWPAZ,0,56.05,Two year,No
3331,7644-OMVMY,0,19.85,Two year,No
3826,3213-VVOLG,0,25.35,Two year,No
4380,2520-SGTTA,0,20.00,Two year,No
5218,2923-ARZLG,0,19.70,One year,No
6670,4075-WKNIU,0,73.35,Two year,No


NaN after conversion: 11
Rows with tenure == 0: 11

--- Target Variable Breakdown ---
       Count  Percentage (%)
Churn                       
No      5174           73.46
Yes     1869           26.54

Imbalance ratio (majority to minority): 2.77x

Numerical columns (3): ['tenure', 'MonthlyCharges', 'TotalCharges']
Categorical columns (16): ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Category levels per column:
  gender: {'Male': 3555, 'Female': 3488}
  SeniorCitizen: {0: 5901, 1: 1142}
  Partner: {'No': 3641, 'Yes': 3402}
  Dependents: {'No': 4933, 'Yes': 2110}
  PhoneService: {'Yes': 6361, 'No': 682}
  MultipleLines: {'No': 3390, 'Yes': 2971, 'No phone service': 682}
  InternetService: {'Fiber optic': 3096, 'DSL': 2421, 'No': 1526}
  OnlineSecurity: {'No': 3498, 'Yes': 201

,count,mean,std,min,25%,50%,75%,max,skewness
tenure,7043.0,32.37,24.56,0.00,9.00,29.00,55.00,72.00,0.24
MonthlyCharges,7043.0,64.76,30.09,18.25,35.50,70.35,89.85,118.75,-0.22
TotalCharges,7032.0,2283.30,2266.77,18.80,401.45,1397.48,3794.74,8684.80,0.96



--- Leakage Check ---
Suspicious columns among features: None

TotalCharges / (tenure x MonthlyCharges): median 1.000, min 0.689, max 1.573

Spearman correlation of numeric features with churn:
tenure           -0.367
MonthlyCharges    0.185
TotalCharges     -0.232
dtype: float64

Churn rate by key categories:
  Contract: {'Month-to-month': 0.427, 'One year': 0.113, 'Two year': 0.028}
  InternetService: {'DSL': 0.19, 'Fiber optic': 0.419, 'No': 0.074}
  PaymentMethod: {'Bank transfer (automatic)': 0.167, 'Credit card (automatic)': 0.152, 'Electronic check': 0.453, 'Mailed check': 0.191}

STAGE 3 — PREPROCESSING, TRAINING, EVALUATION

--- P0 ---
Python 3.14.2 | pandas 3.0.2 | scikit-learn 1.8.0 | joblib 1.5.3

--- P1 ---
Raw shape: (7043, 21)
Clean shape: (7043, 20) | TotalCharges NaN left: 0
Churn
No churn    5174
Churn       1869
Name: count, dtype: int64

--- P2 ---
Train: (5634, 19) | Test: (1409, 19)


,train,test,train %,test %
Churn,,,,
No churn,4139,1035,73.46,73.46
Churn,1495,374,26.54,26.54



--- P3 ---
Logistic Regression  -> prep -> clf
Decision Tree        -> prep -> clf
KNN                  -> prep -> clf
Gaussian NB          -> prep -> clf
SVM                  -> prep -> clf

--- P4 ---
Logistic Regression  trained in 0.07 s | classes: [0, 1] | encoded columns: 40
Decision Tree        trained in 0.07 s | classes: [0, 1] | encoded columns: 40
KNN                  trained in 0.04 s | classes: [0, 1] | encoded columns: 40
Gaussian NB          trained in 0.19 s | classes: [0, 1] | encoded columns: 40
SVM                  trained in 9.86 s | classes: [0, 1] | encoded columns: 40

--- P5 ---


,Accuracy,AUC,Precision,Recall,F1,MCC
Logistic Regression,0.7381,0.8415,0.5043,0.7834,0.6136,0.4531
Decision Tree,0.7360,0.6566,0.5028,0.4866,0.4946,0.3160
KNN,0.7622,0.7766,0.5543,0.5321,0.5430,0.3825
Gaussian NB,0.6849,0.8128,0.4499,0.8396,0.5858,0.4138
SVM,0.7821,0.8219,0.5827,0.6310,0.6059,0.4564



--- P6 ---
Saved models_project_b/logistic_regression.joblib (7.8 KB)
Saved models_project_b/decision_tree.joblib (183.6 KB)
Saved models_project_b/knn.joblib (1790.1 KB)
Saved models_project_b/gaussian_nb.joblib (8.9 KB)
Saved models_project_b/svm.joblib (1018.2 KB)
Saved metadata.json, test_set.csv, metrics_project_b.csv

--- P7 ---
Logistic Regression  reload OK, identical predictions: True
Decision Tree        reload OK, identical predictions: True
KNN                  reload OK, identical predictions: True
Gaussian NB          reload OK, identical predictions: True
SVM                  reload OK, identical predictions: True


In [7]:
# =========================================================
# PROJECT B — Telco Churn: FINAL VALIDATION (single cell)
# Loads the SAVED files only. Nothing is retrained.
# =========================================================
import json
import numpy as np, pandas as pd, sklearn, joblib
from pathlib import Path
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef)

MODEL_DIR = Path("models_project_b")                         # Project B output folder
RAW_CSV = Path("/drive/notebooks") / "Telco-Customer-Churn.csv"   # original dataset
checks = {}                                                  # name -> True/False, summarised at the end

# ---------------------------------------------------------
# 0. METADATA & VERSION
# ---------------------------------------------------------
meta = json.loads((MODEL_DIR / "metadata.json").read_text())
TARGET, FEATURES = meta["target"], meta["feature_columns"]
print(f"Saved with scikit-learn {meta['sklearn_version']} | running scikit-learn {sklearn.__version__}")
checks["0. scikit-learn version matches"] = meta["sklearn_version"] == sklearn.__version__

# ---------------------------------------------------------
# 1. ALL FIVE MODELS LOAD
# ---------------------------------------------------------
print("\n--- 1. Loading saved models ---")
models = {}
for name, fname in meta["models"].items():
    try:
        models[name] = joblib.load(MODEL_DIR / fname)
        print(f"  {name:20s} loaded from {fname}")
    except Exception as e:
        print(f"  {name:20s} FAILED: {e}")
checks["1. all 5 models load"] = len(models) == 5

# ---------------------------------------------------------
# 4. TEST SET HAS ALL FEATURES + TARGET (checked before predicting)
# ---------------------------------------------------------
print("\n--- 4. Test set structure ---")
test = pd.read_csv(MODEL_DIR / "test_set.csv")
X_test, y_test = test[FEATURES], test[TARGET]
print(f"  Shape: {test.shape} | missing values: {int(test.isna().sum().sum())}")
print(f"  Class counts: {y_test.value_counts().sort_index().to_dict()}")
checks["4a. columns = 19 features + target"] = list(test.columns) == FEATURES + [TARGET]
checks["4b. no missing values"] = not test.isna().any().any()
checks["4d. customerID not used as a feature"] = "customerID" not in FEATURES
checks["4e. TotalCharges is numeric"] = pd.api.types.is_numeric_dtype(test["TotalCharges"])
checks["4c. features match what models expect"] = all(list(m.feature_names_in_) == FEATURES for m in models.values())

# ---------------------------------------------------------
# 2 & 3. SAVED MODELS PREDICT + METRICS MATCH THE CSV
# ---------------------------------------------------------
print("\n--- 2 & 3. Predict with saved models and recompute metrics ---")
def evaluate(pipe):
    y_pred, churn_proba = pipe.predict(X_test), pipe.predict_proba(X_test)[:, 1]
    return {"Accuracy": accuracy_score(y_test, y_pred),
            "AUC": roc_auc_score(y_test, churn_proba),
            "Precision": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
            "Recall": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
            "F1": f1_score(y_test, y_pred, pos_label=1, zero_division=0),
            "MCC": matthews_corrcoef(y_test, y_pred)}

recomputed = pd.DataFrame({name: evaluate(pipe) for name, pipe in models.items()}).T
saved = pd.read_csv(MODEL_DIR / "metrics_project_b.csv", index_col=0)
max_diff = (recomputed - saved.loc[recomputed.index, recomputed.columns]).abs().max().max()
display(recomputed.round(4))
print(f"  Largest difference vs metrics_project_b.csv: {max_diff:.2e}")
checks["2. every model predicts on the test set"] = len(recomputed) == 5
checks["3. metrics match the saved CSV"] = max_diff < 1e-9

# ---------------------------------------------------------
# 5. LEAKAGE: rebuild the split and inspect what the pipelines learned
# ---------------------------------------------------------
print("\n--- 5. Leakage checks ---")
df = pd.read_csv(RAW_CSV)
ids = df["customerID"]
df = df.drop(columns=["customerID"])
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
df[TARGET] = (df[TARGET] == "Yes").astype(int)
X, y = df[FEATURES], df[TARGET]
X_train, X_te2, y_train, y_te2 = train_test_split(X, y, test_size=meta["test_size"], stratify=y,
                                                  random_state=meta["random_state"])

same_split = X_te2.reset_index(drop=True).equals(X_test.reset_index(drop=True))
print(f"  Re-created split equals saved test set: {same_split}")
checks["5a. split is reproducible"] = same_split

exact_overlap = len(pd.merge(test, X_train.assign(**{TARGET: y_train}), how="inner"))
feature_overlap = len(pd.merge(X_test, X_train, how="inner"))
print(f"  Test rows also in training (features + label): {exact_overlap}")
print(f"  Test rows whose features appear in training: {feature_overlap} (allowed: different customers with identical plans)")
# Same split applied to customer IDs: no customer may appear on both sides
ids_train, ids_test = train_test_split(ids, test_size=meta["test_size"], stratify=y,
                                       random_state=meta["random_state"])
shared_ids = len(set(ids_train) & set(ids_test))
print(f"  Customers present in both train and test: {shared_ids}")
checks["5b. no customer in both train and test"] = shared_ids == 0

# A scaler fitted on TRAINING data only must have the training means, not the full-data means
NUM_COLS = meta["numeric_columns"]
scaler = models["Logistic Regression"].named_steps["prep"].named_transformers_["num"]
train_ok = np.allclose(scaler.mean_, X_train[NUM_COLS].mean().values)
full_ok = np.allclose(scaler.mean_, X[NUM_COLS].mean().values)
print(f"  Scaler mean = training mean: {train_ok} | = full-data mean: {full_ok}")
checks["5c. preprocessing fitted on training data only"] = train_ok and not full_ok
encoder = models["Logistic Regression"].named_steps["prep"].named_transformers_["cat"]
print(f"  One-hot encoder outputs {len(encoder.get_feature_names_out())} columns from {len(meta['categorical_columns'])} categorical features")

# ---------------------------------------------------------
# SUMMARY
# ---------------------------------------------------------
print("\n" + "=" * 60 + "\nVALIDATION SUMMARY — PROJECT B\n" + "=" * 60)
for name, ok in checks.items():
    print(f"  {'PASS' if ok else 'FAIL'}  {name}")
print("\nALL CHECKS PASSED" if all(checks.values()) else "\nSOME CHECKS FAILED — share this output")


Saved with scikit-learn 1.8.0 | running scikit-learn 1.8.0

--- 1. Loading saved models ---
  Logistic Regression  loaded from logistic_regression.joblib
  Decision Tree        loaded from decision_tree.joblib
  KNN                  loaded from knn.joblib
  Gaussian NB          loaded from gaussian_nb.joblib
  SVM                  loaded from svm.joblib

--- 4. Test set structure ---
  Shape: (1409, 20) | missing values: 0
  Class counts: {0: 1035, 1: 374}

--- 2 & 3. Predict with saved models and recompute metrics ---


,Accuracy,AUC,Precision,Recall,F1,MCC
Logistic Regression,0.7381,0.8415,0.5043,0.7834,0.6136,0.4531
Decision Tree,0.7360,0.6566,0.5028,0.4866,0.4946,0.3160
KNN,0.7622,0.7766,0.5543,0.5321,0.5430,0.3825
Gaussian NB,0.6849,0.8128,0.4499,0.8396,0.5858,0.4138
SVM,0.7821,0.8219,0.5827,0.6310,0.6059,0.4564


  Largest difference vs metrics_project_b.csv: 5.55e-17

--- 5. Leakage checks ---
  Re-created split equals saved test set: True
  Test rows also in training (features + label): 8
  Test rows whose features appear in training: 15 (allowed: different customers with identical plans)
  Customers present in both train and test: 0
  Scaler mean = training mean: True | = full-data mean: False
  One-hot encoder outputs 37 columns from 16 categorical features

VALIDATION SUMMARY — PROJECT B
  PASS  0. scikit-learn version matches
  PASS  1. all 5 models load
  PASS  4a. columns = 19 features + target
  PASS  4b. no missing values
  PASS  4d. customerID not used as a feature
  PASS  4e. TotalCharges is numeric
  PASS  4c. features match what models expect
  PASS  2. every model predicts on the test set
  PASS  3. metrics match the saved CSV
  PASS  5a. split is reproducible
  PASS  5b. no customer in both train and test
  PASS  5c. preprocessing fitted on training data only

ALL CHECKS PASSED
